<a href="https://colab.research.google.com/github/gmauricio-toledo/math-ML/blob/main/Notebooks/22-Autoencoders-Quitar_marcas_de_agua.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instrucciones

1. Completa las celdas que están marcadas en rojo.
2. Ejecuta todas las celdas
3. Realiza la comparación de la última sección y redacta tu respuesta en una celda de texto

In [ ]:
#@title Funciones para crear marcas de agua

import os
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt


def crear_marca_de_agua(ancho: int, alto: int, texto: str = "MUESTRA • ML", opacidad: int = 85) -> Image.Image:
    """
    Genera una capa RGBA transparente con una marca de agua realista estilo stock,
    escalada proporcionalmente al tamaño de la imagen.

    Parámetros:
        ancho, alto: Dimensiones de la imagen destino.
        texto: Texto que aparecerá en el sello central.
        opacidad: Transparencia de 0 (invisible) a 255 (sólido). Recomendado: 70-100.
    """
    capa = Image.new("RGBA", (ancho, alto), (255, 255, 255, 0))
    draw = ImageDraw.Draw(capa)

    # Grosor y tamaño de fuente proporcionales a la dimensión menor
    dim_base = min(ancho, alto)
    grosor_linea = max(1, dim_base // 90)
    tam_fuente = max(12, dim_base // 11)

    color_blanco = (255, 255, 255, opacidad)
    color_sombra = (0, 0, 0, int(opacidad * 0.6))

    # 1. Líneas diagonales cruzadas muy tenues (típicas de bancos de imágenes)
    draw.line([(0, 0), (ancho, alto)], fill=(255, 255, 255, int(opacidad * 0.45)), width=grosor_linea)
    draw.line([(0, alto), (ancho, 0)], fill=(255, 255, 255, int(opacidad * 0.45)), width=grosor_linea)

    # 2. Cargar fuente escalable (compatible con Pillow >= 10.1 y versiones anteriores)
    try:
        fuente = ImageFont.load_default(size=tam_fuente)
    except TypeError:
        fuente = ImageFont.load_default()

    # Calcular caja del texto para centrarlo
    bbox = draw.textbbox((0, 0), texto, font=fuente)
    txt_w = bbox[2] - bbox[0]
    txt_h = bbox[3] - bbox[1]

    cx, cy = ancho // 2, alto // 2
    pad_x = int(txt_w * 0.22)
    pad_y = int(txt_h * 0.45)

    # 3. Marco rectangular redondeado alrededor del texto
    caja_sello = [
        cx - txt_w // 2 - pad_x,
        cy - txt_h // 2 - pad_y,
        cx + txt_w // 2 + pad_x,
        cy + txt_h // 2 + pad_y,
    ]
    radio = max(4, dim_base // 25)

    # Sombra sutil + borde blanco para que sea visible en fondos claros y oscuros
    draw.rounded_rectangle(
        [c + max(1, grosor_linea) for c in caja_sello],
        radius=radio,
        outline=color_sombra,
        width=grosor_linea + 1
    )
    draw.rounded_rectangle(
        caja_sello,
        radius=radio,
        fill=(255, 255, 255, int(opacidad * 0.22)),
        outline=color_blanco,
        width=grosor_linea + 1
    )

    # 4. Texto centrado con ligero sombreado para contraste universal
    pos_texto = (cx - txt_w // 2 - bbox[0], cy - txt_h // 2 - bbox[1])
    draw.text((pos_texto[0] + 1, pos_texto[1] + 1), texto, font=fuente, fill=color_sombra)
    draw.text(pos_texto, texto, font=fuente, fill=color_blanco)

    return capa


def aplicar_marca_de_agua(imagen: Image.Image, texto: str = "MUESTRA • ML", opacidad: int = 85) -> Image.Image:
    """
    Aplica la marca de agua predeterminada sobre una imagen PIL y devuelve la imagen RGB resultante.
    Esta misma función se usará tanto en el dataset como en tus imágenes propias de prueba.
    """
    img_rgba = imagen.convert("RGBA")
    marca = crear_marca_de_agua(img_rgba.width, img_rgba.height, texto=texto, opacidad=opacidad)
    combinada = Image.alpha_composite(img_rgba, marca)
    return combinada.convert("RGB")


def generar_dataset_pares(
    carpeta_origen: str,
    carpeta_limpias: str,
    carpeta_marcadas: str,
    tamaño_salida: tuple[int, int] | None = (128, 128),
    texto_marca: str = "MUESTRA • ML",
    opacidad: int = 85
):
    """
    Lee todas las imágenes JPG/PNG de `carpeta_origen`, las redimensiona (opcional)
    y guarda los pares alineados en `carpeta_limpias` y `carpeta_marcadas`.
    """
    origen = Path(carpeta_origen)
    dir_limpias = Path(carpeta_limpias)
    dir_marcadas = Path(carpeta_marcadas)

    dir_limpias.mkdir(parents=True, exist_ok=True)
    dir_marcadas.mkdir(parents=True, exist_ok=True)

    extensiones = {".jpg", ".jpeg", ".png"}
    rutas = sorted([p for p in origen.iterdir() if p.suffix.lower() in extensiones])

    if not rutas:
        raise FileNotFoundError(f"No se encontraron imágenes JPG o PNG en: {origen}")

    ejemplos_preview = []

    for idx, ruta in enumerate(rutas):
        with Image.open(ruta) as img:
            img_rgb = img.convert("RGB")

            # Estandarizar tamaño facilita mucho la carga de tensores en el autoencoder
            if tamaño_salida is not None:
                img_rgb = img_rgb.resize(tamaño_salida, Image.Resampling.LANCZOS)

            img_con_marca = aplicar_marca_de_agua(img_rgb, texto=texto_marca, opacidad=opacidad)

            # Guardar ambas en PNG sin pérdida para evitar artefactos JPEG extra
            nombre_archivo = f"{ruta.stem}.png"
            img_rgb.save(dir_limpias / nombre_archivo)
            img_con_marca.save(dir_marcadas / nombre_archivo)

            if len(ejemplos_preview) < 3:
                ejemplos_preview.append((img_rgb, img_con_marca, nombre_archivo))

    print(f"Proceso completado: {len(rutas)} pares generados.")
    print(f"  -> Originales limpias: {dir_limpias}")
    print(f"  -> Con marca de agua:  {dir_marcadas}")

    # Visualización rápida en la notebook
    fig, axes = plt.subplots(len(ejemplos_preview), 2, figsize=(8, 3.5 * len(ejemplos_preview)))
    if len(ejemplos_preview) == 1:
        axes = [axes]

    for fila, (limpia, marcada, nombre) in enumerate(ejemplos_preview):
        axes[fila][0].imshow(limpia)
        axes[fila][0].set_title(f"Original ({nombre})")
        axes[fila][0].axis("off")

        axes[fila][1].imshow(marcada)
        axes[fila][1].set_title("Con marca de agua (Entrada)")
        axes[fila][1].axis("off")

    plt.tight_layout()
    plt.show()



In [ ]:
#@title Descargar y organizar dataset
import shutil
from pathlib import Path
import tensorflow as tf

# 1. Descarga del dataset
url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
ruta_extraida = Path(tf.keras.utils.get_file("flower_photos", origin=url, extract=True))

# Compatibilidad entre Keras 2 y Keras 3 con la ruta devuelta al descomprimir
if not ruta_extraida.is_dir():
    ruta_extraida = ruta_extraida.parent

# 2. Reunir 2,500 imágenes en ./dataset_raw (sin subcarpetas)
carpeta_raw = Path("./dataset_raw")
carpeta_raw.mkdir(parents=True, exist_ok=True)

todas_las_imagenes = sorted(ruta_extraida.rglob("*.jpg"))

for idx, img_path in enumerate(todas_las_imagenes[:2500]):
    # Se antepone el nombre de la subcarpeta para evitar colisiones de nombres
    destino = carpeta_raw / f"{img_path.parent.name}_{idx:04d}.jpg"
    shutil.copy2(img_path, destino)

print(f"Listo: {len(list(carpeta_raw.glob('*.jpg')))} imágenes preparadas en {carpeta_raw}")

Poner marcas de agua a las imagenes

In [ ]:
generar_dataset_pares(
    carpeta_origen="./dataset_raw",
    carpeta_limpias="./dataset_proyecto/clean",
    carpeta_marcadas="./dataset_proyecto/watermarked",
    tamaño_salida=(128, 128),  # Usa None si prefieres conservar la resolución original
    texto_marca="UNISON • DEMO",
    opacidad=85
)

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

def cargar_pares_numpy(carpeta_marcadas: str, carpeta_limpias: str):
    rutas_x = sorted(Path(carpeta_marcadas).glob("*.png"))
    rutas_y = sorted(Path(carpeta_limpias).glob("*.png"))

    X = np.array([np.asarray(Image.open(p), dtype=np.float32) / 255.0 for p in rutas_x])
    Y = np.array([np.asarray(Image.open(p), dtype=np.float32) / 255.0 for p in rutas_y])
    return X, Y

# Autoencoder MLP

🔴 Define el autoencoder y compílalo

In [ ]:
autoencoder = ...


autoencoder.summary()

In [ ]:
# Cargar tensores normalizados en [0, 1]
X, Y = cargar_pares_numpy("./dataset_proyecto/watermarked", "./dataset_proyecto/clean")
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.15, random_state=42)

# Entrenamiento (EarlyStopping evita sobreajuste si el dataset es pequeño)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True
)

🔴 Define el entrenamiento y realizalo

In [ ]:
hist_ae = autoencoder.fit(
    # ...
    validation_data=..., # escoge los datos de validación
    epochs=..., # define el número de epocas
    batch_size=16, # define un tamaño de lote
    callbacks=[early_stop], # mantén este callback
    verbose=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(hist_ae.history["loss"], label="train_loss")
plt.plot(hist_ae.history["val_loss"], label="val_loss")
plt.legend()
plt.show()

🔴 Obten el error de reconstrucción de este modelo

# U-Net

In [ ]:
def bloque_conv(x, filtros: int):
    """Bloque doble de convolución estándar en arquitecturas U-Net."""
    x = layers.Conv2D(filtros, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(filtros, (3, 3), padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    return x


def construir_unet(input_shape=(128, 128, 3)) -> tf.keras.Model:
    """
    U-Net ligera con skip connections para reconstrucción de imagen preservando detalles.
    """
    inputs = layers.Input(shape=input_shape)

    # --- ENCODER (Contracting Path) ---
    c1 = bloque_conv(inputs, 32)       # 128x128x32
    p1 = layers.MaxPooling2D((2, 2))(c1)

    c2 = bloque_conv(p1, 64)           # 64x64x64
    p2 = layers.MaxPooling2D((2, 2))(c2)

    c3 = bloque_conv(p2, 128)          # 32x32x128
    p3 = layers.MaxPooling2D((2, 2))(c3)

    c4 = bloque_conv(p3, 256)          # 16x16x256
    p4 = layers.MaxPooling2D((2, 2))(c4)

    # --- BOTTLENECK ---
    b = bloque_conv(p4, 256)           # 8x8x256

    # --- DECODER (Expansive Path + Skip Connections) ---
    u4 = layers.UpSampling2D((2, 2))(b)               # 16x16
    u4 = layers.Concatenate()([u4, c4])               # 16x16x512
    d4 = bloque_conv(u4, 256)

    u3 = layers.UpSampling2D((2, 2))(d4)              # 32x32
    u3 = layers.Concatenate()([u3, c3])               # 32x32x384
    d3 = bloque_conv(u3, 128)

    u2 = layers.UpSampling2D((2, 2))(d3)              # 64x64
    u2 = layers.Concatenate()([u2, c2])               # 64x64x192
    d2 = bloque_conv(u2, 64)

    u1 = layers.UpSampling2D((2, 2))(d2)              # 128x128
    u1 = layers.Concatenate()([u1, c1])               # 128x128x96
    d1 = bloque_conv(u1, 32)

    # Proyección final a 3 canales RGB en [0, 1]
    outputs = layers.Conv2D(3, (1, 1), activation="sigmoid", padding="same")(d1)

    model = models.Model(inputs, outputs, name="UNet_Ligera")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mae")
    return model

In [ ]:
# 2. Instanciar modelos
unet = construir_unet()

In [ ]:
hist_unet = unet.fit(...) # Define los parámetros de entrenamiento

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot(hist_unet.history["loss"], label="train_loss")
plt.plot(hist_unet.history["val_loss"], label="val_loss")
plt.legend()
plt.show()

🔴 Obten el error de reconstrucción de este modelo

# Comparación

In [ ]:
#@title Función para comparar ambas reconstrucciones

import numpy as np
import matplotlib.pyplot as plt


def visualizar_comparacion_grid(X_test, modelo_ae, modelo_unet, n_cols=5, semilla=None):
    """
    Genera una cuadrícula de 3 filas x 5 columnas con imágenes aleatorias de prueba:
      - Fila 1: Originales con marca de agua (Entrada)
      - Fila 2: Reconstrucción con Autoencoder estándar
      - Fila 3: Reconstrucción con U-Net
    """
    if semilla is not None:
        np.random.seed(semilla)

    n_muestras = min(n_cols, len(X_test))
    indices = np.random.choice(len(X_test), size=n_muestras, replace=False)
    muestras_entrada = X_test[indices]

    # Inferencia con ambos modelos
    pred_ae = np.clip(modelo_ae.predict(muestras_entrada, verbose=0), 0.0, 1.0)
    pred_unet = np.clip(modelo_unet.predict(muestras_entrada, verbose=0), 0.0, 1.0)

    filas = [
        ("Con marca de agua\n(Entrada)", muestras_entrada),
        ("Autoencoder\nMLP", pred_ae),
        ("U-Net", pred_unet),
    ]

    fig, axes = plt.subplots(3, n_muestras, figsize=(3.2 * n_muestras, 9.5))

    if n_muestras == 1:
        axes = axes[:, np.newaxis]

    for fila_idx, (etiqueta_fila, imagenes) in enumerate(filas):
        for col_idx in range(n_muestras):
            ax = axes[fila_idx, col_idx]
            ax.imshow(imagenes[col_idx])

            # Ocultar marcas y marco manteniendo el espacio para la etiqueta de fila
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

            if fila_idx == 0:
                ax.set_title(f"Muestra #{indices[col_idx]}", fontsize=11, pad=8)

            if col_idx == 0:
                ax.set_ylabel(
                    etiqueta_fila,
                    fontsize=11,
                    fontweight="bold",
                    rotation=90,
                    labelpad=12,
                    va="center",
                )

    plt.tight_layout()
    plt.show()

In [ ]:
visualizar_comparacion_grid(X_val, autoencoder, unet, n_cols=5, semilla=None)

🔴 Viendo las comparaciones de reconstrucciones, ambas curvas de entrenamiento y ambos errores de reconstrucción, argumenta cuál modelo lo hizo mejor y conjetura porque fue el caso.